In [7]:
!pip install transformers torch pdfplumber


In [8]:
import pdfplumber

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        if len(pdf.pages) <= 5:
            raise ValueError("PDF must contain more than 5 pages")
        for page in pdf.pages:
            text += page.extract_text() + " "
    return text


In [9]:
import re

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()

def chunk_text(text, max_words=500):
    words = text.split()
    return [" ".join(words[i:i+max_words])
            for i in range(0, len(words), max_words)]


In [10]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")


In [21]:
def summarize_chunk(chunk):
    inputs = tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=180,
        min_length=120,
        num_beams=6,
        length_penalty=1.5
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [22]:
intermediate_summary = " ".join(
    summarize_chunk(chunk) for chunk in chunks
)


In [23]:
def compress_to_50_words(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    output_ids = model.generate(
        inputs["input_ids"],
        max_length=70,
        min_length=50,
        num_beams=6
    )
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Enforce exact 50 words
    words = summary.split()
    return " ".join(words[:50])


In [24]:
def generate_complete_50_word_summary(pdf_path):
    text = extract_text(pdf_path)
    text = clean_text(text)
    chunks = chunk_text(text)

    intermediate_summary = " ".join(
        summarize_chunk(chunk) for chunk in chunks
    )

    final_summary = compress_to_50_words(intermediate_summary)
    return final_summary


In [26]:
summary = generate_complete_50_word_summary("Indian Cricket Enters a New Era.pdf")
print("FINAL SUMMARY (≈50 words):\n")
print(summary)
print("\nWord Count:", len(summary.split()))


FINAL SUMMARY (≈50 words):

Cricket experts agree that India’s future depends on long-term vision rather than short-term success. Fast bowlers have consistently delivered match-winning spells, supported by a reliable group of all-rounders. Spin bowling continues to dominate home conditions, providing India with a formidable advantage. Social media has transformed fan engagement

Word Count: 47
